## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-21-15-45-00 +0000,nypost,LA’s dirty secret washes ashore as storms dump...,https://nypost.com/2026/11/21/us-news/la-rains...
1,2026-11-21-15-37-45 +0000,nyt,"It’s a Bill on Energy Permits, but Trump’s Pet...",https://www.nytimes.com/2026/11/20/us/politics...
2,2026-11-21-15-31-25 +0000,nyt,The California Grandparents Accused of Killing...,https://www.nytimes.com/2026/11/21/us/jonathan...
3,2026-11-21-15-29-00 +0000,wsj,Mexico Captures Top Drug Boss in Fierce Gunfight,https://www.wsj.com/world/americas/mexico-capt...
4,2026-11-21-15-24-08 +0000,nypost,"Over 50K units of toothpaste, mouthwash recall...",https://nypost.com/2026/11/21/business/over-50...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2593/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
12,trump,40
256,he,17
130,new,15
602,anthropic,14
409,tech,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
299,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,124
225,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,103
214,2026-11-20-21-58-36 +0000,latimes,Trump says he'll penalize U.S. companies that ...,https://www.latimes.com/politics/story/2026-11...,97
141,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,92
259,2026-11-20-20-14-29 +0000,nypost,Trump backs Jensen Huang’s return as Nvidia CE...,https://nypost.com/2026/11/20/us-news/trump-ba...,89


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
299,124,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
223,53,2026-11-20-21-43-22 +0000,bbc,What the new UK-EU deal could mean for future ...,https://www.bbc.co.uk/news/videos/c90k6gl3lyxe...
141,52,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...
112,48,2026-11-21-05-13-03 +0000,nyt,"Alice Huang, Accused of Spying on Nvidia’s CEO...",https://www.nytimes.com/2026/11/20/us/politics...
253,46,2026-11-20-20-25-49 +0000,nypost,"‘GTA 6’ maker Take-Two rakes in nearly $4B, sh...",https://nypost.com/2026/11/20/business/take-tw...
238,41,2026-11-20-21-06-45 +0000,nypost,Anthropic researcher claims Claude cracked leg...,https://nypost.com/2026/11/20/business/anthrop...
96,37,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...
91,37,2026-11-21-10-00-00 +0000,wapo,5 things you should consider leaving out of yo...,https://www.washingtonpost.com/business/2026/1...
12,33,2026-11-21-15-00-00 +0000,nypost,Texas Democrats push high-speed rail plan afte...,https://nypost.com/2026/11/21/us-news/texas-de...
169,33,2026-11-20-23-12-10 +0000,nypost,Melania Trump attends Fox News ‘All-American C...,https://nypost.com/2026/11/20/us-news/melania-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
